## Reading text-data into Python

In [12]:
# read the file in the project directory
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("total number of character: ", len(raw_text))
# print first 100
print(raw_text[:99])

total number of character:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## Split text to obtain a list of tokens
- split the text on all types of punctuations, whitespace(\s), commas, periods, question marks, quotation marks, double-dashes

In [34]:
import re
# text = "Hello, world. This is a test. Hello, world. This is a test."
text = raw_text

# Because of (), these separators are also included in the result.
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) 

# .strip() removes whitespace from both ends and store in [] 
# if item.strip() decides whether to keep the item, don't keep "<empty_str>"
preprocessed = [item.strip() for item in preprocessed if item.strip()] 

# print(preprocessed)
print(f"the number of tokens in preprocessed text: {len(preprocessed)}")

the number of tokens in preprocessed text: 4690


## Converting tokens into token IDs
- convert these tokens from a __string to an integer__ representation to produce the token IDs
- individual tokens are then sorted alphabetically, and duplicate tokens are removed
- each sorted token is then mapped to an integer.

In [35]:
# sorted set of tokens
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"length of sorted set of tokens: {vocab_size}")

# dictionary of token and corresponding token id
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=10:
        break
        

length of sorted set of tokens: 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


### Complete tokenizer class with encode and decode method
- encode method: splits text into tokens and carries out the string string-to-integer mapping to produce token IDs via the vocabulary
- decode method: carries out reverse integer-to-string mapping to convert the token IDs back into text.
  

In [36]:
class SimpleTokenizerV1:

    # constructor takes a vocab, dictionary of token -> ID
    def __init__(self, vocab):
        # Stores the vocabulary as a class attribute for access in the encode and decode methods
        self.str_to_int = vocab 
        # Creates an inverse vocubalary ID -> token
        self.int_to_str = {i:s for s,i in vocab.items()} 

    # process input text into token
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # convert token IDs back into text
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # "separator".join(list) 
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # r'\1' captured punctuation, replaces " !" -> "!"
        return text
    

In [37]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""

# text = "Hello, do you like tea?" # KeyError, Hello was not used in vocab 

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


### Adding special context tokens
- to handle unkown words, tokenizer needs to be modifies ie. extended vocabulary with additional special tokens
- <|endoftext|> tokens act as markers, signaling the start or end of a particular segment, allowing for more effective processing and understanding by the LLM
- 

In [76]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print("the new vocabulary size: ",len(vocab.items()))
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

the new vocabulary size:  1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


## tokenizer V2 to handle unkown words

In [82]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])',r'\1',text)
        return text
                    

In [88]:
# <\endoftext\> token to concatenate from two unrelated sentences
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1,text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [90]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Byte pair ecoding used by GPT models
- breaks words down into subword units
- implemented here using open source library __tiktoken__
- The BPE tokenizer can handle any unknown word.

In [100]:
!pip install tiktoken

In [96]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [98]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
"Hello, do you like tea? <|endoftext|> In the sunlit terraces"
"of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [99]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## Data sampling with a sliding window 
- the input–target pairs required for training an LLM
- LLMs are pretrained by predicting the next word in a text,
- tokenize the whole "The Verdict" using BPE tokenizer.
- to create input-target pairs for next word prediction, create two variables x and y, x contains input tokens and y contains target.

In [112]:
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    enc_text = tokenizer.encode(raw_text)
# print(len(enc_text))

enc_sample = enc_text[50:]

# context size determines number of tokens are included in the input
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:       {y}")

x: [290, 4920, 2241, 287]
y:       [4920, 2241, 287, 257]


In [121]:
# By processing the inputs along with the targets, which are the inputs shifted by one position,
# create the next-word prediction tasks
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # left of arrow is ? LLM gets and right is ? supposed to predict 
    print(context, "---->", desired) 
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
    print("\n")

[290] ----> 4920
 and ---->  established


[290, 4920] ----> 2241
 and established ---->  himself


[290, 4920, 2241] ----> 287
 and established himself ---->  in


[290, 4920, 2241, 287] ----> 257
 and established himself in ---->  a




### PyTorch built-in Dataset and DataLoader classes

In [125]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 3.7 MB/s  0:00:21 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.0 MB/s  0:00:004.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 3.8 MB/s  0:00:01m 3.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 1.6 MB/s  0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torch]━━━━━ 5/6 [torch]kx]mpy]


In [129]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataSetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1: i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]
    